# Drive-normalized scaling analysis

**Public-release note.** This notebook is part of a GitHub-ready version of the KMC memristor project. Notebook outputs were stripped to keep the repository lightweight, and path settings were adjusted to use repository-relative locations where needed.

**Purpose:** Fits logistic formation curves, extracts E50 and wE, and evaluates the reduced descriptors Delta, E/E50, E-hat, and Psi.

**Main manuscript role:** Used for threshold-normalized formation, switching-time scaling, and branch-topology organization analyses.

**Default assumption:** run the notebook from inside this repository so that the `results/` directory can be discovered automatically.



# KMC drive-normalized scaling（mE fixed-T）
这版 notebook 用来做你现在最需要的 **下一步分析**：

\[
m \;\rightarrow\; \Delta,\; l_c
\quad\text{但 topology 还受 drive 影响}
\]

所以这里不再只看 \((\Delta,l_c)\)，而是加入 **归一化驱动力**：

1. 先对每个 `m` 的 `formed_prob(E)` 拟合 logistic 曲线  
2. 提取：
   - \(E_{50}(m)\)：formation probability = 0.5 时的阈值场
   - \(w_E(m)\)：formation curve 宽度（经验宽度）
3. 定义：
   - \(\tilde E = E / E_{50}(m)\)
   - \(\hat E = (E - E_{50}(m))/w_E(m)\)
4. 重新看 kinetic / topology 是否能在 **drive-normalized** 变量上变得更干净

## 这版重点
- 只主打 **mE fixed-T** 主线
- 图全部用 **seaborn**
- kinetic 继续用 `phase2_mE_mE_summary.csv`
- topology 优先用 `phase2_mE_mE_formed_topology_summary.csv`
- topology 默认优先使用：
  - `stage = crit / first-percolation`
  - `stat = median`

## 你最需要改的地方
最上面这几个参数：

- `BASE_DIR`
- `FIXED_T_FOR_mE`
- `TOPO_STAGE`
- `TOPO_STAT`
- `USE_MID_PFORM_FILTER`
- `ALPHA_LIST`

## 跑完后先看
1. `mE_formation_curves_and_E50_seaborn.png`
2. `mE_formed_prob_vs_Ehat_seaborn.png`
3. `mE_log10_tset_vs_Enorm_seaborn.png`
4. `mE_branches_vs_Enorm_seaborn.png`
5. `mE_branches_in_Delta_Enorm_space_seaborn.png`
6. `best_Psi_mE_branches_seaborn.png`


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import curve_fit


def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in [start] + list(start.parents):
        if (path / "README.md").exists() and (path / "notebooks").exists() and (path / "results").exists():
            return path
    return start

# =========================
# 0) Repository-aware setup
# =========================
REPO_ROOT = find_repo_root()
BASE_DIR = REPO_ROOT / "results" / "core_scans"

# Fixed temperature used when the summary table does not store T explicitly
FIXED_T_FOR_mE = 700.0

# Default topology convention
TOPO_STAGE = "crit"     # "crit" / "lrs"
TOPO_STAT = "median"    # "median" / "mean"

# Optional mid-probability filter for topology analysis
USE_MID_PFORM_FILTER = False
PFORM_MIN = 0.10
PFORM_MAX = 0.90

# Composite descriptor scan:
# Psi_alpha = (E/E50) / [ Delta * (lc/lc0)^alpha ]
ALPHA_LIST = np.linspace(-1.0, 2.5, 71)

OUT_DIR_NAME = "scaling"

SAVE_FIG = True
SHOW_FIG = True


In [ ]:

sns.set_theme(style="whitegrid", context="talk", font_scale=1.0)

KB_EV = 8.617333262145e-5

BASE_DIR = Path(BASE_DIR)
OUT_DIR = REPO_ROOT / "results" / OUT_DIR_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILE_ME_SUM = BASE_DIR / "phase2_mE_mE_summary.csv"
FILE_ME_TOPO = BASE_DIR / "phase2_mE_mE_formed_topology_summary.csv"

print("BASE_DIR =", BASE_DIR)
print("OUT_DIR  =", OUT_DIR)
print()
print("summary file exists:", FILE_ME_SUM.exists(), FILE_ME_SUM)
print("topology file exists:", FILE_ME_TOPO.exists(), FILE_ME_TOPO)


In [ ]:

# =========================
# 1) 工具函数
# =========================
def pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"找不到列名候选：{candidates}\n现有列：{list(df.columns)}")
    return None


def ensure_numeric(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


def logistic_ab(x, a, b):
    return 1.0 / (1.0 + np.exp(-(a + b * x)))


def fit_logistic_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]

    if len(x) < 4:
        return None

    # 如果 formed_prob 全都接近 0 或 1，很难拟合 E50
    if (np.nanmax(y) - np.nanmin(y)) < 0.15:
        return None

    x0 = np.nanmedian(x)
    # 简单初猜：E50 附近 slope 正
    p0 = np.array([-x0 * 50.0, 50.0], dtype=float)

    try:
        popt, pcov = curve_fit(
            logistic_ab, x, y,
            p0=p0,
            maxfev=20000
        )
        a, b = popt
        if abs(b) < 1e-12:
            return None

        y_pred = logistic_ab(x, a, b)
        rmse = float(np.sqrt(np.mean((y - y_pred) ** 2)))
        e50 = float(-a / b)
        wE = float(1.0 / abs(b))
        return {
            "a": float(a),
            "b": float(b),
            "rmse": rmse,
            "E50": e50,
            "wE": wE,
            "x": x,
            "y": y,
            "y_pred": y_pred
        }
    except Exception:
        return None


def interpolate_e50(x, y):
    """拟合失败时的保底方案：线性插值寻找 y=0.5 对应的 x"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]
    if len(x) < 2:
        return np.nan

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    # 找到跨过 0.5 的区间
    for i in range(len(x) - 1):
        y1, y2 = y[i], y[i + 1]
        if (y1 - 0.5) * (y2 - 0.5) <= 0 and abs(y2 - y1) > 1e-12:
            t = (0.5 - y1) / (y2 - y1)
            return float(x[i] + t * (x[i + 1] - x[i]))
    return np.nan


def standardize_mE_summary(df_raw):
    out = pd.DataFrame(index=df_raw.index)

    out["source"] = "mE"
    out["m_target"] = df_raw[pick_col(df_raw, ["m_target", "m", "m_actual_mean", "m_actual"])]
    out["m_actual"] = df_raw[pick_col(df_raw, ["m_actual_mean", "m_actual", "m_target", "m"])]

    e_col = pick_col(df_raw, ["E", "E_vnm", "E_field", "Efield_vnm", "E_fixed", "E_fixed_vnm"])
    out["E"] = df_raw[e_col]

    t_col = pick_col(df_raw, ["T", "T_K", "temp_K", "temperature_K", "T_fixed", "T_fixed_K"], required=False)
    if t_col is not None:
        out["T"] = df_raw[t_col]
    else:
        out["T"] = FIXED_T_FOR_mE

    sigma_col = pick_col(df_raw, ["sigma_E_mean", "sigma_E", "barrier_std_z", "std_em_z"], required=False)
    if sigma_col is not None:
        out["sigma_E"] = df_raw[sigma_col]
    else:
        out["sigma_E"] = np.nan

    lc_col = pick_col(df_raw, ["lc_mean", "l_c", "lc", "l_c_mean"], required=False)
    if lc_col is not None:
        out["lc"] = df_raw[lc_col]
    else:
        out["lc"] = np.nan

    delta_col = pick_col(df_raw, ["Delta_mean", "Delta"], required=False)
    if delta_col is not None:
        out["Delta"] = df_raw[delta_col]
    else:
        out["Delta"] = out["sigma_E"] / (KB_EV * out["T"])

    formed_col = pick_col(df_raw, ["formed_prob", "Pform", "formation_prob"])
    out["formed_prob"] = df_raw[formed_col]

    tset_col = pick_col(df_raw, ["log10_t_set_mean", "log10_t_set", "t_set_log10_mean"], required=False)
    if tset_col is not None:
        out["log10_t_set"] = df_raw[tset_col]
    else:
        out["log10_t_set"] = np.nan

    out = ensure_numeric(out, ["m_target", "m_actual", "E", "T", "sigma_E", "lc", "Delta", "formed_prob", "log10_t_set"])
    return out


def standardize_mE_topology(df_raw, stage="crit", stat="median"):
    out = pd.DataFrame(index=df_raw.index)

    out["source"] = "mE"
    out["m_target"] = df_raw[pick_col(df_raw, ["m_target", "m", "m_actual_mean", "m_actual"])]
    out["m_actual"] = df_raw[pick_col(df_raw, ["m_actual_mean", "m_actual", "m_target", "m"])]

    e_col = pick_col(df_raw, ["E", "E_vnm", "E_field", "Efield_vnm", "E_fixed", "E_fixed_vnm"])
    out["E"] = df_raw[e_col]

    t_col = pick_col(df_raw, ["T", "T_K", "temp_K", "temperature_K", "T_fixed", "T_fixed_K"], required=False)
    if t_col is not None:
        out["T"] = df_raw[t_col]
    else:
        out["T"] = FIXED_T_FOR_mE

    sigma_col = pick_col(df_raw, ["sigma_E_mean", "sigma_E", "barrier_std_z", "std_em_z", "sigma_real_mean"], required=False)
    if sigma_col is not None:
        out["sigma_E"] = df_raw[sigma_col]
    else:
        out["sigma_E"] = np.nan

    lc_col = pick_col(df_raw, ["lc_mean", "l_c", "lc", "l_c_mean"], required=False)
    if lc_col is not None:
        out["lc"] = df_raw[lc_col]
    else:
        out["lc"] = np.nan

    delta_col = pick_col(df_raw, ["Delta_mean", "Delta"], required=False)
    if delta_col is not None:
        out["Delta"] = df_raw[delta_col]
    else:
        out["Delta"] = out["sigma_E"] / (KB_EV * out["T"])

    pform_col = pick_col(df_raw, ["formed_prob", "Pform", "formation_prob"], required=False)
    out["formed_prob"] = df_raw[pform_col] if pform_col is not None else np.nan

    # 优先选新的 crit/lrs + mean/median 列
    stage = stage.lower()
    stat = stat.lower()

    branch_candidates = ["branches_topo_main"]
    tort_candidates = ["tortuosity_topo_main"]
    neck_candidates = ["neck_topo_main"]

    if stage == "crit":
        if stat == "median":
            branch_candidates += ["branches_crit_median_formed"]
            tort_candidates += ["tortuosity_nm_crit_median_formed"]
            neck_candidates += ["neck_nm_crit_median_formed"]
        else:
            branch_candidates += ["branches_crit_mean_formed"]
            tort_candidates += ["tortuosity_nm_crit_mean_formed"]
            neck_candidates += ["neck_nm_crit_mean_formed"]
    elif stage == "lrs":
        if stat == "median":
            branch_candidates += ["branches_lrs_median_formed", "branches_median_formed"]
            tort_candidates += ["tortuosity_nm_lrs_median_formed", "tortuosity_nm_median_formed"]
            neck_candidates += ["neck_nm_lrs_median_formed", "neck_nm_median_formed"]
        else:
            branch_candidates += ["branches_lrs_mean_formed", "branches_mean_formed"]
            tort_candidates += ["tortuosity_nm_lrs_mean_formed", "tortuosity_nm_mean_formed"]
            neck_candidates += ["neck_nm_lrs_mean_formed", "neck_nm_mean_formed"]

    # 兜底
    branch_candidates += ["branches_mean_formed", "branches_median_formed", "branches_mean", "branches"]
    tort_candidates += ["tortuosity_nm_mean_formed", "tortuosity_nm_median_formed", "tortuosity_mean", "tortuosity"]
    neck_candidates += ["neck_nm_mean_formed", "neck_nm_median_formed", "neck_mean", "neck_nm", "neck"]

    out["branches"] = df_raw[pick_col(df_raw, branch_candidates)]
    out["tortuosity"] = df_raw[pick_col(df_raw, tort_candidates)]
    out["neck_nm"] = df_raw[pick_col(df_raw, neck_candidates, required=False)] if any(c in df_raw.columns for c in neck_candidates) else np.nan

    out = ensure_numeric(out, ["m_target", "m_actual", "E", "T", "sigma_E", "lc", "Delta", "formed_prob", "branches", "tortuosity", "neck_nm"])
    return out


def apply_mid_pform_filter(df):
    if not USE_MID_PFORM_FILTER:
        return df.copy()
    if "formed_prob" not in df.columns:
        return df.copy()
    out = df.copy()
    return out[(out["formed_prob"] >= PFORM_MIN) & (out["formed_prob"] <= PFORM_MAX)].copy()


def add_delta_log10_tset_by_m(df):
    out = df.copy()
    out["delta_log10_t_set"] = np.nan

    for mval, g in out.groupby("m_target", dropna=False):
        sub = g.dropna(subset=["log10_t_set"]).copy()
        if len(sub) == 0:
            continue
        ref = sub["log10_t_set"].min()
        out.loc[sub.index, "delta_log10_t_set"] = sub["log10_t_set"] - ref
    return out


def fit_e50_by_m(df):
    rows = []
    fit_objs = {}

    sub_all = df.dropna(subset=["m_target", "E", "formed_prob"]).copy()

    for mval, g in sub_all.groupby("m_target", dropna=False):
        g = g.sort_values("E")
        fit = fit_logistic_1d(g["E"].values, g["formed_prob"].values)

        if fit is None:
            e50_fallback = interpolate_e50(g["E"].values, g["formed_prob"].values)
            rows.append({
                "m_target": mval,
                "n_points": len(g),
                "fit_ok": False,
                "a": np.nan,
                "b": np.nan,
                "rmse": np.nan,
                "E50": e50_fallback,
                "wE": np.nan,
            })
            fit_objs[mval] = None
        else:
            rows.append({
                "m_target": mval,
                "n_points": len(g),
                "fit_ok": True,
                "a": fit["a"],
                "b": fit["b"],
                "rmse": fit["rmse"],
                "E50": fit["E50"],
                "wE": fit["wE"],
            })
            fit_objs[mval] = fit

    e50_df = pd.DataFrame(rows).sort_values("m_target").reset_index(drop=True)
    return e50_df, fit_objs


def merge_drive_descriptors(df, e50_df):
    out = df.merge(e50_df[["m_target", "E50", "wE", "fit_ok"]], on="m_target", how="left")
    out["E_norm"] = out["E"] / out["E50"]
    out["E_hat"] = (out["E"] - out["E50"]) / out["wE"]
    return out


def fit_rmse_quadratic(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]
    if len(x) < 4:
        return np.nan, np.nan, None, None

    X = np.column_stack([np.ones_like(x), x, x**2])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    y_pred = X @ coef
    rmse = np.sqrt(np.mean((y - y_pred) ** 2))
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2) + 1e-12
    r2 = 1 - ss_res / ss_tot
    return rmse, r2, coef, y_pred


def scan_alpha_for_psi(df, ycol, alpha_list=None):
    if alpha_list is None:
        alpha_list = ALPHA_LIST

    sub = df.dropna(subset=["E_norm", "Delta", "lc", ycol]).copy()
    if len(sub) < 4:
        return pd.DataFrame()

    lc_ref = float(np.nanmedian(sub["lc"].values))
    rows = []

    for alpha in alpha_list:
        Psi = sub["E_norm"].values / (sub["Delta"].values * ((sub["lc"].values / lc_ref) ** alpha))
        rmse, r2, coef, y_pred = fit_rmse_quadratic(Psi, sub[ycol].values)
        rows.append({
            "alpha": alpha,
            "lc_ref": lc_ref,
            "rmse": rmse,
            "r2": r2,
        })

    return pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)


In [ ]:

# =========================
# 2) 读入并标准化
# =========================
df_me_sum_raw = pd.read_csv(FILE_ME_SUM)
df_me_topo_raw = pd.read_csv(FILE_ME_TOPO)

df_me_sum = standardize_mE_summary(df_me_sum_raw)
df_me_sum = add_delta_log10_tset_by_m(df_me_sum)

df_me_topo = standardize_mE_topology(df_me_topo_raw, stage=TOPO_STAGE, stat=TOPO_STAT)
df_me_topo = apply_mid_pform_filter(df_me_topo)

print("mE summary columns:")
print(df_me_sum.columns.tolist())
print("\nmE topology columns:")
print(df_me_topo.columns.tolist())
print("\nshapes:", df_me_sum.shape, df_me_topo.shape)

df_me_sum.to_csv(OUT_DIR / "standardized_mE_summary.csv", index=False)
df_me_topo.to_csv(OUT_DIR / "standardized_mE_topology_summary.csv", index=False)


In [ ]:

# =========================
# 3) 拟合每个 m 的 formation curve，提取 E50 / wE
# =========================
e50_df, fit_objs = fit_e50_by_m(df_me_sum)
display(e50_df)
e50_df.to_csv(OUT_DIR / "mE_E50_fit_table.csv", index=False)

# 画 formation curves
plt.figure(figsize=(8.2, 6.2))
pal = sns.color_palette("viridis", n_colors=max(3, df_me_sum["m_target"].nunique()))
for i, (mval, g) in enumerate(df_me_sum.dropna(subset=["m_target", "E", "formed_prob"]).groupby("m_target")):
    color = pal[i % len(pal)]
    g = g.sort_values("E")
    sns.scatterplot(
        data=g, x="E", y="formed_prob",
        color=color, s=80, label=f"m={mval:.2f}"
    )
    fit = fit_objs.get(mval, None)
    if fit is not None:
        xx = np.linspace(g["E"].min(), g["E"].max(), 200)
        yy = logistic_ab(xx, fit["a"], fit["b"])
        plt.plot(xx, yy, color=color, lw=2)
        plt.axvline(fit["E50"], color=color, ls="--", lw=1, alpha=0.6)

plt.xlabel("E (V/nm)")
plt.ylabel("formed_prob")
plt.title("mE: formation curves and extracted E50(m)")
plt.legend(ncol=2, fontsize=10)
if SAVE_FIG:
    plt.savefig(OUT_DIR / "mE_formation_curves_and_E50_seaborn.png", dpi=220, bbox_inches="tight")
if SHOW_FIG:
    plt.show()
else:
    plt.close()


In [ ]:

# =========================
# 4) 合并 drive-normalized descriptor
# =========================
df_me_sum_dn = merge_drive_descriptors(df_me_sum, e50_df)
df_me_topo_dn = merge_drive_descriptors(df_me_topo, e50_df)

# 只保留 E50 有意义的点
df_me_sum_dn = df_me_sum_dn[np.isfinite(df_me_sum_dn["E50"]) & (df_me_sum_dn["E50"] > 0)].copy()
df_me_topo_dn = df_me_topo_dn[np.isfinite(df_me_topo_dn["E50"]) & (df_me_topo_dn["E50"] > 0)].copy()

display(df_me_sum_dn.head())
display(df_me_topo_dn.head())

df_me_sum_dn.to_csv(OUT_DIR / "mE_drive_normalized_kinetic.csv", index=False)
df_me_topo_dn.to_csv(OUT_DIR / "mE_drive_normalized_topology.csv", index=False)


In [ ]:

# =========================
# 5) kinetic：drive-normalized scaling
# =========================
# formed_prob vs E_hat
sub = df_me_sum_dn.dropna(subset=["E_hat", "formed_prob"]).copy()

plt.figure(figsize=(6.2, 4.8))
sns.scatterplot(data=sub, x="E_hat", y="formed_prob", hue="m_target", palette="viridis", s=90)
plt.xlabel(r"$\hat E = (E - E_{50})/w_E$")
plt.ylabel("formed_prob")
plt.title("mE: formation probability vs normalized drive")
if SAVE_FIG:
    plt.savefig(OUT_DIR / "mE_formed_prob_vs_Ehat_seaborn.png", dpi=220, bbox_inches="tight")
if SHOW_FIG:
    plt.show()
else:
    plt.close()

# log10_t_set vs E_norm
sub = df_me_sum_dn.dropna(subset=["E_norm", "log10_t_set"]).copy()

plt.figure(figsize=(6.2, 4.8))
sns.scatterplot(data=sub, x="E_norm", y="log10_t_set", hue="Delta", palette="magma", s=90)
plt.xlabel(r"$\tilde E = E/E_{50}$")
plt.ylabel("log10_t_set")
plt.title("mE: switching delay vs drive-normalized field")
if SAVE_FIG:
    plt.savefig(OUT_DIR / "mE_log10_tset_vs_Enorm_seaborn.png", dpi=220, bbox_inches="tight")
if SHOW_FIG:
    plt.show()
else:
    plt.close()

# delta_log10_t_set vs E_norm
sub = df_me_sum_dn.dropna(subset=["E_norm", "delta_log10_t_set"]).copy()

plt.figure(figsize=(6.2, 4.8))
sns.scatterplot(data=sub, x="E_norm", y="delta_log10_t_set", hue="m_target", palette="viridis", s=90)
plt.xlabel(r"$\tilde E = E/E_{50}$")
plt.ylabel(r"$\delta \log_{10}(t_{set})$")
plt.title("mE: relative switching delay vs drive-normalized field")
if SAVE_FIG:
    plt.savefig(OUT_DIR / "mE_delta_log10_tset_vs_Enorm_seaborn.png", dpi=220, bbox_inches="tight")
if SHOW_FIG:
    plt.show()
else:
    plt.close()


In [ ]:

# =========================
# 6) topology：先看 E_norm，再看 2D 描述
# =========================
# branches vs E_norm
sub = df_me_topo_dn.dropna(subset=["E_norm", "branches"]).copy()

plt.figure(figsize=(6.2, 4.8))
sns.scatterplot(data=sub, x="E_norm", y="branches", hue="Delta", size="lc", palette="magma", sizes=(60, 220))
plt.xlabel(r"$\tilde E = E/E_{50}$")
plt.ylabel("branches")
plt.title(f"mE topology ({TOPO_STAGE}, {TOPO_STAT}): branches vs normalized drive")
if SAVE_FIG:
    plt.savefig(OUT_DIR / "mE_branches_vs_Enorm_seaborn.png", dpi=220, bbox_inches="tight")
if SHOW_FIG:
    plt.show()
else:
    plt.close()

# tortuosity vs E_norm
sub = df_me_topo_dn.dropna(subset=["E_norm", "tortuosity"]).copy()

plt.figure(figsize=(6.2, 4.8))
sns.scatterplot(data=sub, x="E_norm", y="tortuosity", hue="Delta", size="lc", palette="magma", sizes=(60, 220))
plt.xlabel(r"$\tilde E = E/E_{50}$")
plt.ylabel("tortuosity")
plt.title(f"mE topology ({TOPO_STAGE}, {TOPO_STAT}): tortuosity vs normalized drive")
if SAVE_FIG:
    plt.savefig(OUT_DIR / "mE_tortuosity_vs_Enorm_seaborn.png", dpi=220, bbox_inches="tight")
if SHOW_FIG:
    plt.show()
else:
    plt.close()

# branches in (Delta, E_norm)
sub = df_me_topo_dn.dropna(subset=["Delta", "E_norm", "branches"]).copy()

plt.figure(figsize=(6.2, 4.8))
sns.scatterplot(data=sub, x="Delta", y="E_norm", hue="branches", size="lc", palette="plasma", sizes=(60, 220))
plt.xlabel(r"$\Delta = \sigma_E/(k_B T)$")
plt.ylabel(r"$\tilde E = E/E_{50}$")
plt.title("mE topology: branches in (Delta, E/E50) space")
if SAVE_FIG:
    plt.savefig(OUT_DIR / "mE_branches_in_Delta_Enorm_space_seaborn.png", dpi=220, bbox_inches="tight")
if SHOW_FIG:
    plt.show()
else:
    plt.close()

# tortuosity in (Delta, E_norm)
sub = df_me_topo_dn.dropna(subset=["Delta", "E_norm", "tortuosity"]).copy()

plt.figure(figsize=(6.2, 4.8))
sns.scatterplot(data=sub, x="Delta", y="E_norm", hue="tortuosity", size="lc", palette="plasma", sizes=(60, 220))
plt.xlabel(r"$\Delta = \sigma_E/(k_B T)$")
plt.ylabel(r"$\tilde E = E/E_{50}$")
plt.title("mE topology: tortuosity in (Delta, E/E50) space")
if SAVE_FIG:
    plt.savefig(OUT_DIR / "mE_tortuosity_in_Delta_Enorm_space_seaborn.png", dpi=220, bbox_inches="tight")
if SHOW_FIG:
    plt.show()
else:
    plt.close()


In [ ]:

# =========================
# 7) 组合 descriptor：Psi_alpha
# Psi_alpha = (E/E50) / [ Delta * (lc/lc0)^alpha ]
# =========================
score_br = scan_alpha_for_psi(df_me_topo_dn, ycol="branches", alpha_list=ALPHA_LIST)
score_to = scan_alpha_for_psi(df_me_topo_dn, ycol="tortuosity", alpha_list=ALPHA_LIST)

display(score_br.head(10))
display(score_to.head(10))

score_br.to_csv(OUT_DIR / "alpha_scan_Psi_branches.csv", index=False)
score_to.to_csv(OUT_DIR / "alpha_scan_Psi_tortuosity.csv", index=False)


In [ ]:

def plot_best_psi_fit(df, ycol, alpha_best, out_png, title):
    sub = df.dropna(subset=["E_norm", "Delta", "lc", ycol]).copy()
    if len(sub) < 4:
        print(f"[skip] {title}: 数据不足")
        return None

    lc_ref = float(np.nanmedian(sub["lc"].values))
    sub["Psi"] = sub["E_norm"] / (sub["Delta"] * ((sub["lc"] / lc_ref) ** alpha_best))

    rmse, r2, coef, y_pred = fit_rmse_quadratic(sub["Psi"].values, sub[ycol].values)

    plt.figure(figsize=(6.2, 4.8))
    sns.scatterplot(data=sub, x="Psi", y=ycol, hue="m_target", palette="viridis", s=90)

    if coef is not None:
        xx = np.linspace(np.nanmin(sub["Psi"]), np.nanmax(sub["Psi"]), 300)
        yy = coef[0] + coef[1] * xx + coef[2] * xx**2
        plt.plot(xx, yy, color="black", lw=2, label=f"quad fit, R²={r2:.3f}")
        plt.legend()

    plt.xlabel(r"$\Psi_\alpha = (E/E_{50}) / [\Delta (l_c/l_{c,0})^\alpha]$" + f"\n(alpha={alpha_best:.2f})")
    plt.ylabel(ycol)
    plt.title(title)

    if SAVE_FIG:
        plt.savefig(OUT_DIR / out_png, dpi=220, bbox_inches="tight")
    if SHOW_FIG:
        plt.show()
    else:
        plt.close()

    return {"alpha": alpha_best, "rmse": rmse, "r2": r2}


best_br = None
best_to = None

if len(score_br) > 0:
    alpha_best_br = float(score_br.iloc[0]["alpha"])
    best_br = plot_best_psi_fit(
        df_me_topo_dn,
        ycol="branches",
        alpha_best=alpha_best_br,
        out_png="best_Psi_mE_branches_seaborn.png",
        title=f"mE topology ({TOPO_STAGE}, {TOPO_STAT}): best Psi collapse for branches"
    )

if len(score_to) > 0:
    alpha_best_to = float(score_to.iloc[0]["alpha"])
    best_to = plot_best_psi_fit(
        df_me_topo_dn,
        ycol="tortuosity",
        alpha_best=alpha_best_to,
        out_png="best_Psi_mE_tortuosity_seaborn.png",
        title=f"mE topology ({TOPO_STAGE}, {TOPO_STAT}): best Psi collapse for tortuosity"
    )


In [ ]:

# =========================
# 8) 输出简洁报告
# =========================
report_rows = []

for _, row in e50_df.iterrows():
    report_rows.append({
        "analysis": "E50_fit_by_m",
        "m_target": row["m_target"],
        "fit_ok": row["fit_ok"],
        "E50": row["E50"],
        "wE": row["wE"],
        "rmse": row["rmse"],
    })

if best_br is not None:
    report_rows.append({
        "analysis": "topology_branches_best_Psi",
        "alpha_best": best_br["alpha"],
        "r2_best": best_br["r2"],
        "rmse_best": best_br["rmse"],
        "stage": TOPO_STAGE,
        "stat": TOPO_STAT,
    })

if best_to is not None:
    report_rows.append({
        "analysis": "topology_tortuosity_best_Psi",
        "alpha_best": best_to["alpha"],
        "r2_best": best_to["r2"],
        "rmse_best": best_to["rmse"],
        "stage": TOPO_STAGE,
        "stat": TOPO_STAT,
    })

report_df = pd.DataFrame(report_rows)
display(report_df)
report_df.to_csv(OUT_DIR / "drive_normalized_scaling_report.csv", index=False)

print(f"\n所有输出都保存在：\n{OUT_DIR}")



## 你现在应该先改哪里
最重要的 5 个输入：

1. `BASE_DIR`  
   你的 phase2 输出目录

2. `FIXED_T_FOR_mE`  
   当前 mE 主线固定温度

3. `TOPO_STAGE` / `TOPO_STAT`  
   当前推荐：
   - `TOPO_STAGE = "crit"`
   - `TOPO_STAT = "median"`

4. `USE_MID_PFORM_FILTER`  
   如果点太少就先关掉：
   ```python
   USE_MID_PFORM_FILTER = False
   ```

5. `ALPHA_LIST`  
   组合 descriptor 的扫描范围

---

## 跑完后先看什么
优先看：

- `mE_formation_curves_and_E50_seaborn.png`
- `mE_formed_prob_vs_Ehat_seaborn.png`
- `mE_log10_tset_vs_Enorm_seaborn.png`
- `mE_branches_vs_Enorm_seaborn.png`
- `mE_branches_in_Delta_Enorm_space_seaborn.png`
- `best_Psi_mE_branches_seaborn.png`

---

## 怎样判断这一步有没有价值
如果你看到：

1. `formed_prob` 用 `E_hat` 比原始 `E` 更干净  
2. `log10_t_set` 用 `E/E50` 更有规律  
3. `branches` / `tortuosity` 在 `(Delta, E/E50)` 空间比原来 `(Delta, lc)` 更有结构  
4. `best_Psi` 的 `R²` 比你之前单纯的 `Xi_alpha` 更高

那就说明你现在这一步是对的：

> topology 不是只由 disorder 决定，还必须结合 normalized drive。

到那一步，dynamic m 才值得作为下一层验证引入。
